# DSA4262 • Logistic regression for m6A prediction

## Goal and workflow
Use the outputs from `m6a_explore_and_clean.ipynb` to train a reproducible site-level logistic-regression baseline.

**Training genes:** five-fold stratified group cross-validation for hyperparameter selection → **validation genes:** choose a classification threshold → **test genes:** final evaluation only.

Includes a dummy baseline, imbalance-aware tuning, fold diagnostics, ROC/precision–recall curves, confusion matrices, calibration diagnostics, coefficients, and saved model/predictions. No actual dataset is included; run this notebook on Ronin where your cleaned files are stored. Results are not yet established for your data.

**Input:** the earlier notebook's `clean_labeled_sites.csv.gz`, `feature_columns.json`, `run_manifest.json`, and optionally `split_assignments.csv.gz` from the **same output folder**. This notebook does not directly parse raw `data.json`.

**Source:** `handout_project2_RNAModifications.html`, Task 1 and format requirements, plus the feature schema of the preceding cleaning notebook. The project requires ROC AUC and PR evaluation, a simple baseline, and comparison to m6Anet; this notebook covers the logistic-regression component, not running m6Anet.

## Setup
Open this in VS Code with your Ronin Python kernel. Change `INPUT_DIR` below, then run cells in order. First complete a full cleaning run (`MAX_LINES=None`). Raw and cleaned inputs remain unchanged; each run creates a new output folder.

In [ ]:
# Uncomment once if needed, then restart the kernel.
# %pip install "scikit-learn>=1.3,<2" pandas numpy matplotlib joblib ipykernel
from pathlib import Path
from datetime import datetime, timezone
import json, sys, hashlib, warnings, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, auc, accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay, brier_score_loss, log_loss)
from sklearn.calibration import calibration_curve
from sklearn.exceptions import ConvergenceWarning
plt.rcParams.update({'figure.figsize': (8,4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'logistic': '#356995', 'dummy': '#8a8a8a'}

### 1. Configure inputs
Set `N_JOBS=2` for modest Ronin instances; more concurrent fits use more memory. The search has 10 candidates × 5 folds = **50 fits**, plus refits. `class_weight='balanced'` is recalculated from each training fold's labels.

This notebook compares unweighted and balanced training, without oversampling validation/test data. No SMOTE is needed for this baseline. The primary tuning score is **average precision (AP)**, a step-weighted PR summary; trapezoidal PR AUC is reported separately.

In [ ]:
INPUT_DIR = Path('/CHANGE/TO/outputs/m6a_eda/YOUR_RUN_FOLDER')
OUTPUT_ROOT = Path.cwd() / 'outputs' / 'logistic_regression'
SEED = 42
N_FOLDS = 5
N_JOBS = 2
C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0]
MAX_ITER = 5000

for name in ('clean_labeled_sites.csv.gz','feature_columns.json','run_manifest.json'):
    if not (INPUT_DIR/name).is_file(): raise FileNotFoundError(f'Check INPUT_DIR: missing {INPUT_DIR/name}')
source_manifest = json.loads((INPUT_DIR/'run_manifest.json').read_text())
if 'max_lines' not in source_manifest or source_manifest['max_lines'] is not None:
    raise ValueError('Use a full cleaning run with MAX_LINES=None; a file prefix is not a final training dataset.')
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Saving to:', RUN_DIR)

### 2. Load and check the feature table
Identifiers are retained for joining and grouping, not supplied as predictors. Missing numeric values are imputed inside each fold; the original cleaner normally produces finite summaries. All-missing columns are rejected so the feature schema cannot silently change. No rows are silently dropped here.

In [ ]:
KEY = ['transcript_id','transcript_position']
features = json.loads((INPUT_DIR/'feature_columns.json').read_text())
df = pd.read_csv(INPUT_DIR/'clean_labeled_sites.csv.gz', dtype={'transcript_id':str,'gene_id':str})
expected_numeric = [f'{loc}_{measure}_{stat}' for stat in ('mean','std','median','q25','q75')
                    for loc in ('minus1','center','plus1') for measure in ('dwell','signal_sd','signal_mean')]
expected = set(expected_numeric + ['n_reads_valid','central_5mer'])
if len(features)!=len(set(features)) or set(features)!=expected:
    raise ValueError('Feature schema differs from the previous notebook; inspect feature_columns.json.')
if not set(KEY+['gene_id','label']+features).issubset(df.columns):
    raise ValueError('Required columns are missing.')
if df.empty or df[KEY+['gene_id','label']].isna().any().any():
    raise ValueError('Need nonempty data and complete site keys, labels and gene IDs.')
if df.duplicated(KEY).any(): raise ValueError('Duplicate site keys; fix upstream.')
if not df.label.isin([0,1]).all() or df.label.nunique()!=2:
    raise ValueError('Need binary labels with both classes present.')
if df.gene_id.str.strip().eq('').any(): raise ValueError('Empty gene IDs prevent safe grouping.')
if df.groupby('transcript_id').gene_id.nunique().max()>1:
    raise ValueError('A transcript maps to multiple genes; resolve before splitting.')
numeric_features = expected_numeric + ['n_reads_valid']
categorical_features = ['central_5mer']
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors='raise')
if np.isinf(df[numeric_features].to_numpy()).any(): raise ValueError('Infinite features: fix upstream.')
if df[numeric_features].isna().all().any(): raise ValueError('A feature is entirely missing.')
if df.central_5mer.isna().any(): raise ValueError('Missing central motifs: check upstream cleaning.')
df['label'] = df.label.astype(int)
print(f'{len(df):,} sites; {df.gene_id.nunique():,} genes; {len(features)} predictors before encoding')
display(df.groupby('label').size().rename('sites').to_frame())

### 3. Reuse or create the common split
Existing split assignments are strictly checked against every site and label. If absent, create an approximately 60/20/20 gene-group split with a fixed seed and save it for the team. Site proportions are approximate because genes have different numbers of sites. Reuse these assignments for all models.

The outer split is not stratified. Inspect the counts; very small or highly imbalanced datasets may require a different pre-agreed splitting design. Do not choose a split based on test performance.

In [ ]:
split_path = INPUT_DIR/'split_assignments.csv.gz'
if split_path.is_file():
    assignments = pd.read_csv(split_path, dtype={'transcript_id':str,'gene_id':str})
    if not set(KEY+['gene_id','label','split']).issubset(assignments.columns):
        raise ValueError('Split file is missing required columns.')
    if assignments.duplicated(KEY).any(): raise ValueError('Duplicate split keys.')
    check = df[KEY+['gene_id','label']].merge(assignments[KEY+['gene_id','label','split']],
        on=KEY, how='outer', suffixes=('_data','_split'), indicator=True, validate='one_to_one')
    if not check['_merge'].eq('both').all(): raise ValueError('Split and data sites do not match exactly.')
    if not (check.gene_id_data.eq(check.gene_id_split) & check.label_data.eq(check.label_split)).all():
        raise ValueError('Split labels or genes differ from the data.')
    df = df.merge(assignments[KEY+['split']], on=KEY, validate='one_to_one')
    split_origin = str(split_path.resolve())
else:
    first = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEED)
    tv, test = next(first.split(df, groups=df.gene_id))
    second = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED)
    tr, va = next(second.split(df.iloc[tv], groups=df.iloc[tv].gene_id))
    df['split'] = 'test'
    df.loc[df.index[tv[tr]], 'split'] = 'train'
    df.loc[df.index[tv[va]], 'split'] = 'validation'
    split_origin = 'Created in this notebook'
if set(df.split.dropna()) != {'train','validation','test'} or df.split.isna().any():
    raise ValueError('Expected exactly train, validation and test assignments.')
if df.groupby('gene_id')['split'].nunique().max()!=1: raise ValueError('Gene leakage across partitions.')
if (df.groupby('split').label.nunique()!=2).any(): raise ValueError('Each partition needs both classes.')
train = df.loc[df.split.eq('train')].reset_index(drop=True)
valid = df.loc[df.split.eq('validation')].reset_index(drop=True)
test = df.loc[df.split.eq('test')].reset_index(drop=True)
split_summary = df.groupby('split').agg(sites=('label','size'), genes=('gene_id','nunique'),
                                       positives=('label','sum'), positive_fraction=('label','mean'))
display(split_summary)
df[KEY+['gene_id','label','split']].to_csv(RUN_DIR/'split_assignments.csv.gz',index=False)

### 4. Build leakage-safe preprocessing and five folds
Numeric predictors: median imputation → standardisation. Motif: one-hot encoding, with unseen categories handled at prediction time. Preprocessing is fitted only on each fold's training portion.

`StratifiedGroupKFold` attempts to preserve class balance while keeping genes disjoint; exact balance is not guaranteed. Each fold is checked for both classes and gene overlap. All five folds are within the **training** partition; neither validation nor test enters tuning.

In [ ]:
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
                         ('scale', StandardScaler())])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_features),
    ('motif', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])
model = Pipeline([('preprocess', preprocessor),
                  ('clf', LogisticRegression(solver='liblinear', max_iter=MAX_ITER, random_state=SEED))])
# liblinear uses L2 regularisation by default; C is inverse regularisation strength.
X_train, y_train = train[features], train.label
if train.gene_id.nunique()<N_FOLDS: raise ValueError('Fewer than five training genes.')
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(cv.split(X_train, y_train, groups=train.gene_id))
fold_rows=[]
fold_ids=np.zeros(len(train),dtype=int)
for fold, (tr, va) in enumerate(folds,1):
    assert set(train.iloc[tr].gene_id).isdisjoint(train.iloc[va].gene_id)
    if y_train.iloc[tr].nunique()!=2 or y_train.iloc[va].nunique()!=2:
        raise ValueError(f'Fold {fold} has one class. Revise grouping/fold count before tuning.')
    if X_train.iloc[tr][numeric_features].isna().all().any():
        raise ValueError(f'Fold {fold} has an all-missing numeric training feature.')
    fold_ids[va]=fold
    fold_rows.append({'fold':fold,'training_sites':len(tr),'held_out_sites':len(va),
                     'held_out_genes':train.iloc[va].gene_id.nunique(),
                     'held_out_positive_fraction':y_train.iloc[va].mean()})
display(pd.DataFrame(fold_rows))
fold_assignments=train[KEY+['gene_id','label']].copy(); fold_assignments['cv_fold']=fold_ids
fold_assignments.to_csv(RUN_DIR/'training_cv_folds.csv.gz',index=False)

### 5. Tune C and imbalance handling
Smaller `C` means stronger regularisation. Compare `class_weight=None` against `'balanced'`. Balanced weighting gives each class equal total weight **within the fit's training data**; it does not make the dataset balanced or guarantee calibrated probabilities.

Select the highest mean five-fold AP; also record ROC AUC. CV scores are tuning estimates, not an independent final evaluation. `error_score='raise'` makes failed fits visible. Investigate any convergence warnings; increase `MAX_ITER` and rerun tuning if necessary.

In [ ]:
search = GridSearchCV(model,
    param_grid={'clf__C': C_VALUES, 'clf__class_weight': [None, 'balanced']},
    scoring={'ap':'average_precision', 'roc_auc':'roc_auc'}, refit='ap',
    cv=folds, n_jobs=N_JOBS, pre_dispatch=N_JOBS, error_score='raise', return_train_score=False,
    verbose=1)
search.fit(X_train, y_train)
best_model = search.best_estimator_  # already refitted on the entire training partition
if best_model.named_steps['clf'].n_iter_.max()>=MAX_ITER:
    raise RuntimeError('Selected model reached MAX_ITER; resolve convergence before evaluation.')
cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_ap')
display(cv_results[['param_clf__C','param_clf__class_weight','mean_test_ap','std_test_ap',
                    'mean_test_roc_auc','std_test_roc_auc','rank_test_ap']])
print('Selected parameters:', search.best_params_)
print('Mean training-CV AP:', search.best_score_)
cv_results.to_csv(RUN_DIR/'hyperparameter_search.csv',index=False)
fig, ax=plt.subplots()
for weight, style in [(None,'-o'),('balanced','--s')]:
    mask=cv_results.param_clf__class_weight.map(lambda x: x is None if weight is None else x==weight)
    part=cv_results.loc[mask].sort_values('param_clf__C')
    ax.errorbar(part.param_clf__C.astype(float),part.mean_test_ap,yerr=part.std_test_ap,
                fmt=style,label=str(weight),capsize=3)
ax.set(xscale='log',xlabel='C (inverse regularisation strength)',ylabel='Mean five-fold AP',
       title='Hyperparameter tuning on training genes')
ax.legend(title='Class weight'); fig.tight_layout(); plt.show()
# Error bars are fold standard deviations, not confidence intervals.

### 6. Choose a threshold on validation data only
ROC AUC and AP use continuous scores and do not need a threshold. For precision, recall and F1, select the validation threshold maximising F1. This treats precision and recall symmetrically; choose a different criterion **before** evaluation if your use case needs it. Ties favour the higher threshold.

Threshold-optimised validation F1 is optimistic because validation selected it. The test partition provides the final check. We do not refit on validation after threshold selection, so the evaluated model and score scale remain unchanged.

In [ ]:
validation_scores=best_model.predict_proba(valid[features])[:,1]
p,r,thresholds=precision_recall_curve(valid.label, validation_scores)
f1_values=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(thresholds),
                    where=(p[:-1]+r[:-1])>0)
best_indices=np.flatnonzero(f1_values==f1_values.max())
chosen_threshold=float(thresholds[best_indices[-1]])
threshold_table=pd.DataFrame({'threshold':thresholds,'precision':p[:-1],'recall':r[:-1],'f1':f1_values})
threshold_table.to_csv(RUN_DIR/'validation_threshold_search.csv',index=False)
print('Chosen validation threshold:',chosen_threshold)
print('Validation AP:',average_precision_score(valid.label,validation_scores))
fig,ax=plt.subplots()
ax.plot(thresholds,f1_values,color=COLORS['logistic'])
ax.axvline(chosen_threshold,color='black',linestyle='--',label=f'Selected: {chosen_threshold:.3f}')
ax.set(xlabel='Probability threshold',ylabel='Validation F1',title='Threshold selection on validation genes')
ax.legend(); fig.tight_layout(); plt.show()

### 7. Final held-out evaluation and baseline
Run after all choices above are fixed. Do not retune from test results. The dummy model predicts the training positive fraction for every site. Its AP equals test prevalence, making it a useful reference for imbalance.

Report both AP and trapezoidal PR AUC with their exact names. Accuracy alone can conceal poor minority-class performance. Brier score/log loss and the reliability curve assess probability quality; weighting can change calibration. No post-hoc calibration is fitted here.

In [ ]:
def evaluate(y, scores, threshold):
    predicted=(scores>=threshold).astype(int)
    precision,recall,_=precision_recall_curve(y,scores)
    tn,fp,fn,tp=confusion_matrix(y,predicted,labels=[0,1]).ravel()
    return {'threshold':threshold,'roc_auc':roc_auc_score(y,scores),
            'average_precision':average_precision_score(y,scores),
            'pr_auc_trapezoid':auc(recall,precision),
            'accuracy':accuracy_score(y,predicted),'balanced_accuracy':balanced_accuracy_score(y,predicted),
            'precision':precision_score(y,predicted,zero_division=0),
            'recall':recall_score(y,predicted,zero_division=0),
            'specificity':tn/(tn+fp), 'f1':f1_score(y,predicted,zero_division=0),
            'brier':brier_score_loss(y,scores),'log_loss':log_loss(y,scores,labels=[0,1]),
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}
dummy=DummyClassifier(strategy='prior').fit(np.zeros((len(train),1)),train.label)
test_scores=best_model.predict_proba(test[features])[:,1]
dummy_scores=dummy.predict_proba(np.zeros((len(test),1)))[:,1]
assert np.isfinite(test_scores).all() and ((test_scores>=0)&(test_scores<=1)).all()
metrics=pd.DataFrame([
    {'model':'Logistic, threshold 0.5',**evaluate(test.label,test_scores,.5)},
    {'model':'Logistic, validation threshold',**evaluate(test.label,test_scores,chosen_threshold)},
    {'model':'Dummy prior, threshold 0.5',**evaluate(test.label,dummy_scores,.5)}
]).set_index('model')
display(metrics.round(4).T)
metrics.to_csv(RUN_DIR/'held_out_test_metrics.csv')
print(f'Test sites: {len(test):,}; test genes: {test.gene_id.nunique():,}; prevalence: {test.label.mean():.2%}')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for name,scores,color in [('Logistic',test_scores,COLORS['logistic']),('Dummy prior',dummy_scores,COLORS['dummy'])]:
    fpr,tpr,_=roc_curve(test.label,scores)
    precision,recall,_=precision_recall_curve(test.label,scores)
    axes[0].plot(fpr,tpr,color=color,label=f'{name}: AUC={roc_auc_score(test.label,scores):.3f}')
    axes[1].step(recall[::-1],precision[::-1],where='pre',color=color,
                 label=f'{name}: AP={average_precision_score(test.label,scores):.3f}')
axes[0].plot([0,1],[0,1],':',color='black')
axes[1].axhline(test.label.mean(),linestyle=':',color='black',label='Test prevalence')
axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='Held-out test ROC')
axes[1].set(xlabel='Recall',ylabel='Precision',title='Held-out test precision–recall')
for ax in axes: ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1.02)
fig.tight_layout(); fig.savefig(RUN_DIR/'test_roc_pr.png',dpi=150); plt.show()

fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,t,title in zip(axes,[.5,chosen_threshold],['Threshold = 0.5','Validation-selected threshold']):
    ConfusionMatrixDisplay.from_predictions(test.label,(test_scores>=t).astype(int),labels=[0,1],
        display_labels=['Unmodified','Modified'],ax=ax,colorbar=False,cmap='Blues')
    ax.set_title(title)
fig.tight_layout(); fig.savefig(RUN_DIR/'test_confusion_matrices.png',dpi=150); plt.show()

observed,predicted=calibration_curve(test.label,test_scores,n_bins=10,strategy='quantile')
fig,ax=plt.subplots()
ax.plot(predicted,observed,'o-',color=COLORS['logistic'],label='Logistic regression')
ax.plot([0,1],[0,1],'--',color='black',label='Perfect calibration')
ax.set(xlabel='Mean predicted probability',ylabel='Observed modified fraction',
       title='Held-out reliability diagram (quantile bins)',xlim=(0,1),ylim=(0,1))
ax.legend(); fig.tight_layout(); fig.savefig(RUN_DIR/'test_calibration.png',dpi=150); plt.show()

### 8. Inspect coefficients
Numeric coefficients refer to one training-standard-deviation increase after imputation/scaling. Correlated summary features can distribute or destabilise coefficients, so these are model associations, not biological causal effects. All motif categories are encoded under regularisation; their coefficients are not simple reference-category effects.

In [ ]:
names=best_model.named_steps['preprocess'].get_feature_names_out()
coefficients=pd.DataFrame({'feature':names,'coefficient':best_model.named_steps['clf'].coef_[0]})
coefficients['abs_coefficient']=coefficients.coefficient.abs()
coefficients=coefficients.sort_values('abs_coefficient',ascending=False)
display(coefficients.head(15))
coefficients.to_csv(RUN_DIR/'model_coefficients.csv',index=False)
fig,ax=plt.subplots(figsize=(10,6))
top=coefficients.head(15).sort_values('coefficient')
ax.barh(top.feature,top.coefficient,color=COLORS['logistic']); ax.axvline(0,color='black',linewidth=.8)
ax.set(title='Largest logistic-regression coefficients',xlabel='Coefficient (log-odds scale)',ylabel='')
fig.tight_layout(); fig.savefig(RUN_DIR/'coefficients.png',dpi=150); plt.show()

### 9. Save the complete fitted pipeline and reproducibility record
The saved pipeline includes the imputer, scaler, encoder and classifier; never preprocess new data separately using newly fitted transformations. Load joblib files only from trusted sources.

`test_site_scores.csv` follows the handout's three-column prediction schema but contains **only the held-out subset**, so it is not the leaderboard submission. For new raw JSON, apply the exact upstream feature extraction first and handle every required site, including QC failures, under a documented policy. The classifier alone cannot reconstruct raw-read features.

In [ ]:
joblib.dump(best_model,RUN_DIR/'logistic_regression_pipeline.joblib')
for subset,scores,name in [(valid,validation_scores,'validation'),(test,test_scores,'test')]:
    predictions=subset[KEY+['gene_id','label']].copy()
    predictions['score']=scores
    predictions['predicted_label']=(scores>=chosen_threshold).astype(int)
    predictions.to_csv(RUN_DIR/f'{name}_predictions_with_labels.csv.gz',index=False)
test_submission=test[KEY].copy(); test_submission['score']=test_scores
test_submission.to_csv(RUN_DIR/'test_site_scores.csv',index=False)

def sha256_file(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()
metadata={
    'input_dir':str(INPUT_DIR.resolve()),'source_cleaning_manifest':source_manifest,
    'clean_table_sha256':sha256_file(INPUT_DIR/'clean_labeled_sites.csv.gz'),
    'features':features,'numeric_features':numeric_features,'categorical_features':categorical_features,
    'seed':SEED,'n_folds':N_FOLDS,'split_origin':split_origin,
    'split_sha256':sha256_file(RUN_DIR/'split_assignments.csv.gz'),
    'C_values':C_VALUES,'class_weights_searched':[None,'balanced'],
    'solver':'liblinear','regularisation':'L2','max_iter':MAX_ITER,
    'best_params':search.best_params_,'best_training_cv_AP':float(search.best_score_),
    'threshold':chosen_threshold,'threshold_rule':'maximum validation F1; higher threshold on ties',
    'fitted_on':'training partition only; validation used for threshold; test used only for evaluation',
    'python':sys.version,'packages':{p:importlib.metadata.version(p) for p in
                                    ('scikit-learn','pandas','numpy','matplotlib','joblib')}
}
(RUN_DIR/'model_metadata.json').write_text(json.dumps(metadata,indent=2))
split_summary.to_csv(RUN_DIR/'split_summary.csv')
print('Saved model, results and plots to:',RUN_DIR.resolve())
# Reuse later:
# loaded = joblib.load(RUN_DIR/'logistic_regression_pipeline.joblib')
# new_scores = loaded.predict_proba(new_clean_site_table[features])[:, 1]

## Interpretation and next steps
- Report held-out ROC AUC, **average precision**, trapezoidal PR AUC, test prevalence, and precision/recall/F1 at the validation-selected threshold. Keep the 0.5 result for comparison.
- Compare all team models on the same saved partitions. Differences in cleaning, sites or splits make direct performance comparisons unreliable.
- Five-fold CV here selects hyperparameters; it is not nested CV. A separate held-out test set supplies the final estimate. Fold standard deviations are not confidence intervals. Sites within genes are correlated.
- Assess whether the signal summaries add value beyond motif and read coverage in a separate, pre-planned ablation using training/validation data. Do not repeatedly use this test set for feature decisions.
- Hct116-only evaluation does not establish generalisation to new cell lines. Inspect SG-NEx distribution shifts and compare against m6Anet as the project requires.
- Once evaluation is complete, you may train a separate deployment model on more labelled data, but do not attribute this test score or threshold automatically to that newly fitted model.
- Record AI assistance and your actual verification steps in the project report. No performance values are claimed before you run your data.

**Validation status:** code is prepared for the previous notebook's schema. Real-data execution, convergence and output plots must be checked on your machine using **Restart Kernel → Run All**. Output directories are suitable for `.gitignore`; commit source code and documentation to the project repository.